<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/dynatree-cre/notebooks/m7_5_colab_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M7.5 — Improved Retrieval Space (Random Projection + Analytic Classifier + Relation Description) & `DT-C4.5` (Colab GPU, Seed 2021)
- **Part A (Controls không train prompt — chạy cực nhanh ~2 phút/lượt):**
  - `DT-C0.5a`: Identity Analytic ($M=1536$, $\lambda=10$)
  - `DT-C0.5b`: RanPAC Random-ReLU Analytic ($M=4096$, $\lambda=10$)
  - `DT-C0.5c`: Random-ReLU ($M=4096$) + Relation Description ($\beta=0.25$)
- **Part B (`DT-C2.5` Flat vs `DT-C4.5` Tree với Improved Retrieval + Candidate Reranking `--rerank-sweep`):**
  - FewRel Standard, Exp B (`semantic_homogeneous`), Exp C (`cross_task_dispersed`), và TACRED.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chon GPU runtime (T4/L4/A100)!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/dynatree-cre'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scipy scikit-learn tabulate pytest
!python scripts/setup_protocol_v1_data.py

gdrive_runs = Path('/content/drive/MyDrive/TAPTA_DynaTree_CRE/runs/dynatree_cre')
gdrive_runs.mkdir(parents=True, exist_ok=True)
local_runs = Path('/content/research-code-lab/runs/dynatree_cre')
local_runs.parent.mkdir(parents=True, exist_ok=True)
if local_runs.is_symlink():
    local_runs.unlink()
elif local_runs.exists():
    shutil.copytree(local_runs, gdrive_runs, dirs_exist_ok=True)
    shutil.rmtree(local_runs)
local_runs.symlink_to(gdrive_runs)
!pytest tests/dynatree_cre -q
print('Setup & Drive backup ready ->', gdrive_runs)

In [ ]:
# Part A — M7.5 Sequential Analytic Controls (FewRel + TACRED, no prompt training)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c0_5_analytic_control_fewrel_5shot_seed2021.yaml \
    --analytic-projection-mode identity \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_5a_identity/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c0_5_analytic_control_fewrel_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c0_5_analytic_control_fewrel_5shot_seed2021.yaml \
    --use-relation-description --description-weight 0.25 \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_5c_randproj4k_desc/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

# TACRED DT-C0.5a/b/c controls (epochs=0, ~40s/run)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_5_tree_analytic_rerank_tacred_5shot_seed2021.yaml \
    --routing-strategy none --inference-mode soft_mix --epochs-override 0 \
    --analytic-projection-mode identity --description-weight 0.0 \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_5a_identity/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_5_tree_analytic_rerank_tacred_5shot_seed2021.yaml \
    --routing-strategy none --inference-mode soft_mix --epochs-override 0 \
    --description-weight 0.0 \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_5b_randproj4k/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_5_tree_analytic_rerank_tacred_5shot_seed2021.yaml \
    --routing-strategy none --inference-mode soft_mix --epochs-override 0 \
    --use-relation-description --description-weight 0.25 \
    --output-dir runs/dynatree_cre/m7_5_dt_c0_5c_randproj4k_desc/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download

In [ ]:
# Part B — DT-C2.5 (Flat) vs DT-C4.5 (Tree) tren FewRel Std, Exp B, Exp C & TACRED
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c2_5_flat_analytic_rerank_fewrel_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_5_tree_analytic_rerank_fewrel_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

for dist, tag in [('semantic_homogeneous', 'expB'), ('cross_task_dispersed', 'expC')]:
    !python experiments/run_dynatree_cre.py \
        --config configs/dynatree_cre/m7_5_dt_c4_5_tree_analytic_rerank_fewrel_5shot_seed2021.yaml \
        --task-distribution {dist} \
        --output-dir runs/dynatree_cre/m7_5_dt_c4_5_{tag}/fewrel_5shot/seed_2021 \
        --rerank-sweep --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_5_dt_c4_5_tree_analytic_rerank_tacred_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

In [ ]:
# Part C — Summary Table & Export m7_5_colab_results.zip
import json
import os
import shutil
from pathlib import Path
from tabulate import tabulate
from google.colab import files

runs_root = Path('/content/research-code-lab/runs/dynatree_cre')
bundle = Path('/content/m7_5_bundle')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)

rows = []
for m_path in sorted(runs_root.glob('m7_5_*/**/metrics.json')):
    run_dir = m_path.parent
    rel = run_dir.relative_to(runs_root)
    shutil.copytree(run_dir, bundle / rel, ignore=shutil.ignore_patterns('*.pt'), dirs_exist_ok=True)
    d = json.loads(m_path.read_text())
    res = json.loads((run_dir / 'resource_metrics.json').read_text())
    sw_path = run_dir / 'rerank_sweep.json'
    best_s = 'N/A'
    if sw_path.exists():
        sw = json.loads(sw_path.read_text())
        bc = max(sw['cells'], key=lambda c: c['final_ncm_whole_accuracy'])
        best_s = f"{bc['final_ncm_whole_accuracy']*100:.2f}% (K={bc['k']}, {bc['fusion_mode']})"
    mem_mb = res['memory_bytes']['total_persistent_state_bytes'] / (1024**2)
    rows.append([
        rel.parts[0],
        d.get('dataset'),
        d.get('task_distribution'),
        f"{d.get('final_ncm_whole_accuracy', 0)*100:.2f}%",
        f"{d.get('ncm_summary', {}).get('average_forgetting', 0)*100:.2f}%",
        best_s,
        f"{mem_mb:.1f} MB",
        f"{res.get('final_inference_latency_ms_per_sample', 0):.1f}ms",
    ])

print(tabulate(rows, headers=['Run', 'Dataset', 'Dist', 'Main NCM', 'NCM F', 'Best Sweep', 'PersistMem', 'Lat'], tablefmt='grid'))
shutil.make_archive('/content/m7_5_colab_results', 'zip', bundle)
print('Bundle size KB:', os.path.getsize('/content/m7_5_colab_results.zip') / 1024)
files.download('/content/m7_5_colab_results.zip')